In [1]:
#! pip install langchain-community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.2.16-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_classic-1.0.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached sqlalchemy-2.0.47-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached aiohttp-3.13.3-cp312-cp312-win_amd64.whl.metadata (8.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached langsmith-0.7.7-py3-none-any.whl.metadata (15 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached numpy-2.4.2-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-win_amd64.whl.metadata (21 kB)

In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader("data/", glob="**/*.txt", loader_cls=TextLoader)
kb_docs = loader.load()



In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter1 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

chunks = text_splitter1.split_documents(kb_docs)

In [5]:
print("Number of Documents:", len(kb_docs))
print()
print("Number of Chunks:", len(chunks))

Number of Documents: 3

Number of Chunks: 388


In [1]:
#! pip install langchain-chroma
#! pip install langchain-openai

In [29]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

f = open("keys/keys.txt")
OPENAI_API_KEY = f.read()
embedding_model = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

# Initialize the database connection
# If database exist, it will connect with the collection_name and persist_directory
# Otherwise a new collection will be created
db = Chroma(collection_name="vector_database", 
            embedding_function=embedding_model, 
            persist_directory="./chroma_db_")

In [7]:
# adding documents to vector DB
db.add_documents(documents=chunks)

['322bd9aa-a411-41ee-8e5f-ad58a1599004',
 '5cf5e3f1-c039-4262-a611-98aa29a50e8d',
 '2b1c0ad3-4969-408e-aa14-ac40dcf47986',
 '8d7412fd-58a0-4eaa-8071-6ae69fc64d1a',
 'bea4950f-dbc9-490d-8f6d-427a58040d66',
 '0d461126-d87a-4862-989f-40bc033c8d56',
 '95ba9b4a-5eb0-4531-a62e-cb470a6ae53a',
 '73aa747e-d5e8-489a-bde5-e09a1ad61f47',
 'ce93de2b-0143-4cac-879c-c6f59b958d2e',
 'ef9efb96-bc40-46fe-b1ef-72c7dca98f44',
 'c71a3229-a68a-456f-83ea-9b252f9c855e',
 '54817794-73bc-4023-a4b2-7214ae81c832',
 'c0b84275-6dad-401f-bd4d-cc1705deb6b4',
 '489c8888-f297-4591-81db-79ece97d0c40',
 'fab85dec-ada3-4860-b8af-ede644f19071',
 '0624d18b-1f86-4603-b008-793b0557b6a8',
 '536d7427-7170-486f-8f90-9e6da2c0e343',
 'c6b1c917-4b5f-4d81-b4ea-fe6c57bfc937',
 'e9821975-1b83-46fd-a068-7648f10c8e07',
 '8bb83b8b-808d-44fd-8e47-5ea8f6d65cb3',
 'def3f36a-417f-4d6f-a5fd-77e73dfc5123',
 'f960bb3e-d869-4b40-9444-4c571996ac6c',
 'd42b80e2-bebf-4cca-b56b-40b44525cd88',
 '8df0ab5a-dbfd-4fe1-a0ce-e2d0c6b7b5de',
 'b7c88c59-44c9-

In [9]:
# We can check the already existing values
print(len(db.get()["ids"]))

776


In [10]:
# Create a Retriever Object 

retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [11]:
# Initialize a Chat Prompt Template

from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
    ]
)

In [12]:
prompt_template

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the following context:\n{context}\nAnswer the question based on the above context: {question}.\nProvide a detailed answer.\nDon’t justify your answers.\nDon’t give information not mentioned in the CONTEXT INFORMATION.\nDo not say "according to the context" or "mentioned in the context" or similar.\n'), additional_kwargs={})])

In [13]:
# Initialize a Generator (i.e. Chat Model)
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(api_key=OPENAI_API_KEY)

In [14]:
#initialize output parser
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [15]:
generator_chain = prompt_template | chat_model | parser

In [16]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [17]:
# Define a RAG Chain

from langchain_core.runnables import RunnablePassthrough

rag_chain = {
    "context": retriever | format_docs, 
    "question": RunnablePassthrough()
} | generator_chain

In [30]:
query = 'What is Alzheimer’s disease?'

rag_chain.invoke(query)

"Alzheimer's disease is a neurodegenerative disease and is the most common form of dementia, accounting for around 60-70% of cases."

In [31]:
query = 'How is Alzheimer’s disease diagnosed?'

rag_chain.invoke(query)

"Alzheimer's disease can be diagnosed using cognitive tests such as the mini-mental state examination (MMSE). One method involves providing instructions for copying drawings as part of the evaluation process."

In [32]:
query = 'What role does beta-amyloid play in Alzheimer’s disease?'

rag_chain.invoke(query)

"Beta-amyloid plays a central role in the pathogenesis of Alzheimer's disease."

In [33]:
query = ' Does education level influence risk?'

rag_chain.invoke(query)

"Yes, higher education and occupational attainment are factors that influence the likelihood of being associated with Alzheimer's."

In [34]:
query = ' What is a common cause of death in Alzheimer’s patients?'

rag_chain.invoke(query)

"Complications from Alzheimer's disease are a common cause of death in Alzheimer's patients."